# 3D Focus Stack (Phase 3)

Tier 1 matplotlib 3D view of normalized aerial-image contrast through defocus.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from src.mask import MaskGrid, kirchhoff_mask, line_space_pattern
from src.pupil import PupilSpec
from src.aerial import aerial_image
from src.dof import focus_stack_contrast
from src import constants as C

In [ ]:
grid = MaskGrid(nx=256, ny=64, pixel_size=2e-9)
mask = kirchhoff_mask(line_space_pattern(grid, pitch_m=40e-9))
defocus_values = np.linspace(-80e-9, 80e-9, 9)
spec = PupilSpec(grid_size=256, na=C.NA_HIGH, obscuration_ratio=0.0)

samples = focus_stack_contrast(mask, grid, defocus_values, pupil_spec=spec, anamorphic=False)
print([(round(s.defocus_m * 1e9, 1), round(s.normalized_contrast, 3)) for s in samples])

In [ ]:
x_nm = None
rows = []
for dz in defocus_values:
    image, wafer = aerial_image(
        mask,
        grid,
        pupil_spec=PupilSpec(grid_size=256, na=C.NA_HIGH, obscuration_ratio=0.0, defocus_m=float(dz)),
        anamorphic=False,
    )
    line = image[wafer.ny // 2, :]
    rows.append(line / max(float(np.max(line)), 1e-12))
    if x_nm is None:
        x_nm = (np.arange(wafer.nx) - wafer.nx / 2) * wafer.pixel_x_m * 1e9

X, Z = np.meshgrid(x_nm, defocus_values * 1e9)
Y = np.array(rows)

fig = plt.figure(figsize=(9, 5))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Z, Y, cmap='viridis', linewidth=0, antialiased=True, alpha=0.9)
ax.set_xlabel('x [nm]')
ax.set_ylabel('defocus [nm]')
ax.set_zlabel('normalized intensity')
ax.set_title('Aerial line through focus')
fig.tight_layout()
plt.show()